In [ ]:
import numpy as np
import pandas as pd
import ast
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
movies = pd.read_csv("/content/drive/MyDrive/Dataset/Movie_Recommender_Dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("/content/drive/MyDrive/Dataset/Movie_Recommender_Dataset/tmdb_5000_credits.csv")

In [ ]:
movies.head(2)

In [ ]:
movies.shape

In [ ]:
credits.head()



In [ ]:
movies = movies.merge(credits, on="title")

In [ ]:
movies.head(1)

In [ ]:
movies = movies[["genres", "id", "keywords", "title", "overview", "cast", "crew"]]

In [ ]:
movies.head()

In [ ]:
def convert(obj):
  L = []
  for i in ast.literal_eval(obj):
    L.append(i["name"])
  return L

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies["genres"] = movies["genres"].apply(convert)
movies.head()

In [ ]:
movies["keywords"] = movies["keywords"].apply(convert)
movies.head()

In [ ]:
def convert3(obj):
  L = []
  counter = 0
  for i in ast.literal_eval(obj):
    if counter != 3:
      L.append(i["name"])
      counter += 1
  return L

In [ ]:
movies["cast"] = movies["cast"].apply(convert3)
movies.head()

In [ ]:
movies['cast'] = movies['cast'].apply(lambda x:x[0:3])

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
#movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.sample(5)

In [ ]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [ ]:
movies.head()

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])
#new.head()

In [ ]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))
new.head()

In [ ]:
# ps = PorterStemmer()

In [ ]:
# def stem(text):
#   y = []
#   # for i in text.split():
#     y.append(ps.stem(i))
#   return " ".join(y)

In [ ]:
# new_df["tags"] = new_df["tags"].apply(stem)

In [ ]:
# new_df["tags"][0]

In [ ]:
# new_df["tags"][1]

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words="english")

In [ ]:
vectors = cv.fit_transform(new["tags"]).toarray()

In [ ]:
vectors.shape

In [ ]:
cv.get_feature_names_out()

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
similarity

In [ ]:
new[new['title'] == 'The Lego Movie'].index[0]

In [ ]:
def recommend(movie):
    index = new[new['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
    for i in distances[1:6]:
        print(new.iloc[i[0]].title)

In [ ]:
recommend("Avatar")

In [ ]:
joblib.dump(new, open("movies.joblib", "wb"))
joblib.dump(similarity, open("similarity.joblib", "wb"))